# 02 · Extract Business, Risk Factors and MD&A text from each 10-K

**Reads** `top100_comp_sec_filings` · **Writes** `filings_10K_text`

The parsing logic lives in `trading_sentiment/filing_parser.py` (unit-tested). Filings whose sections cannot be
found end up with empty strings and are removed by the dbt staging models (`stg_clean_10Ks`, `stg_cleaned_filings`).
This notebook replaces the old `parse_filings` notebook and its `top100_clean_filing` table.

In [ ]:
# Make the repository's Python package importable (Databricks Git folder: cwd is this notebook's folder)
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from trading_sentiment import table

In [ ]:
dbutils.widgets.text("sec_user_agent", "", "SEC User-Agent (name + e-mail)")

from trading_sentiment.edgar import require_user_agent
USER_AGENT = require_user_agent(dbutils.widgets.get("sec_user_agent"))

# Companies whose filings the parser cannot split (non-standard item headings). Extend as needed.
EXCLUDED_COMPANIES = ["COSTCO WHOLESALE CORP /NEW"]

In [ ]:
from pyspark.sql import functions as F

filings = (spark.table(table("top100_comp_sec_filings"))
                .where(F.col("form_type") == "10-K")
                .where(~F.col("company_name").isin(EXCLUDED_COMPANIES)))
print(filings.count(), "10-K filings to parse")

Executors cannot import this repository's package, so the modules used inside the UDF are pickled **by value**.
(Without this the UDF fails with `ModuleNotFoundError: trading_sentiment` on the workers.)

In [ ]:
from pyspark import cloudpickle
from pyspark.sql.types import StringType, StructField, StructType

import trading_sentiment, trading_sentiment.filing_parser as filing_parser
for module in (trading_sentiment, filing_parser):
    cloudpickle.register_pickle_by_value(module)

def _parse(url):
    _full, business, risk, mda = filing_parser.parse_filing_url(url, USER_AGENT)
    return business, risk, mda

parse_udf = F.udf(_parse, StructType([
    StructField("bus_text", StringType()),
    StructField("risks_text", StringType()),
    StructField("mda_text", StringType()),
]))

parsed = (filings.withColumn("parsed", parse_udf("filing_url"))
                 .select("cik", "company_name", "tickers", "form_type", "filing_date", "accessionNumber",
                         "market_cap", "sector", "filing_url", "parsed.*"))
parsed.write.format("delta").mode("overwrite").saveAsTable(table("filings_10K_text"))

## Quality check: how many sections were found?

In [ ]:
text = spark.table(table("filings_10K_text"))
display(text.select(
    F.count("*").alias("filings"),
    F.sum((F.length("bus_text") > 5000).cast("int")).alias("business_ok"),
    F.sum((F.length("risks_text") > 5000).cast("int")).alias("risks_ok"),
    F.sum((F.length("mda_text") > 5000).cast("int")).alias("mda_ok"),
))

Next: run dbt staging (`dbt build --select staging` in `dbt_models/`), then notebooks 03–07.